In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/gegeorge250/grimm-brothers-preprocessing/pg2591-LINES.csv
/kaggle/input/notebooks/gegeorge250/grimm-brothers-preprocessing/pg2591-DOCS.csv
/kaggle/input/notebooks/gegeorge250/grimm-brothers-preprocessing/__results__.html
/kaggle/input/notebooks/gegeorge250/grimm-brothers-preprocessing/pg2591.txt
/kaggle/input/notebooks/gegeorge250/grimm-brothers-preprocessing/__notebook__.ipynb
/kaggle/input/notebooks/gegeorge250/grimm-brothers-preprocessing/__output__.json
/kaggle/input/notebooks/gegeorge250/grimm-brothers-preprocessing/custom.css


## Set Up and Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import re

In [3]:
nltk_resources = [
    'tokenizers/punkt', 
    'taggers/averaged_perceptron_tagger', 
    'corpora/stopwords', 
    'help/tagsets']


In [4]:
for resource in nltk_resources:
    try:
        nltk.data.find(resource)
    except IndexError:
        nltk.download(resource)

In [5]:
nltk.download('averaged_perceptron_tagger_eng')


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [6]:
data_home = "../input"
output_dir = "../working"
csv_file  = f"{output_dir}/p2591-TOKENS.csv" # The file we will create

In [7]:
OHCO = ['doc_title','para_num','sentence_num','token_num']

In [8]:
DOCS = pd.read_csv('/kaggle/input/notebooks/gegeorge250/grimm-brothers-preprocessing/pg2591-DOCS.csv')

In [9]:
LINES = pd.read_csv('/kaggle/input/notebooks/gegeorge250/grimm-brothers-preprocessing/pg2591-LINES.csv')
LINES.head()

,line_num,line_str
0,0,﻿The Project Gutenberg eBook of Grimms' Fairy ...
1,1,NaN
2,2,This eBook is for the use of anyone anywhere i...
3,3,most other parts of the world at no cost and w...
4,4,"whatsoever. You may copy it, give it away or r..."


'THE WEDDING OF MRS FOX'

In [10]:
DOCS = pd.read_csv('/kaggle/input/notebooks/gegeorge250/grimm-brothers-preprocessing/pg2591-DOCS.csv')
DOCS.drop(index=54,inplace=True)
DOCS['doc_title']= DOCS['doc_title'].str.replace('FIRST STORY', "THE WEDDING OF MRS. FOX - PART I")
DOCS['doc_title']= DOCS['doc_title'].str.replace('SECOND STORY', "THE WEDDING OF MRS. FOX - PART II")
DOCS.tail(10)

,doc_id,doc_title,doc_str
52,53,DOCTOR KNOWALL,\n\nThere was once upon a time a poor peasant ...
53,54,THE SEVEN RAVENS,"\n\nThere was once a man who had seven sons, a..."
55,56,THE WEDDING OF MRS. FOX - PART I,\nThere was once upon a time an old fox with n...
56,57,THE WEDDING OF MRS. FOX - PART II,"\nWhen old Mr Fox was dead, the wolf came as a..."
57,58,THE SALAD,\n\nAs a merry young huntsman was once going b...
58,59,THE STORY OF THE YOUTH WHO WENT FORTH TO LEARN...,"\n\nA certain father had two sons, the elder o..."
59,60,KING GRISLY-BEARD,\n\nA great king of a land far away in the Eas...
60,61,IRON HANS,\n\nThere was once upon a time a king who had ...
61,62,CAT-SKIN,"\n\nThere was once a king, whose queen had hai..."
62,63,SNOW-WHITE AND ROSE-RED,\n\nThere was once a poor widow who lived in a...


In [11]:
STORIES = DOCS.copy()
STORIES.set_index('doc_title',inplace=True)
STORIES.head()

,doc_id,doc_str
doc_title,,
THE GOLDEN BIRD,1,"\n\nA certain king had a beautiful garden, and..."
HANS IN LUCK,2,\n\nSome men are born to good luck: all they d...
JORINDA AND JORINDEL,3,"\n\nThere was once an old castle, that stood i..."
THE TRAVELLING MUSICIANS,4,\n\nAn honest farmer had once an ass that had ...
OLD SULTAN,5,"\n\nA shepherd had a faithful dog, called Sult..."


In [12]:
STORIES['doc_str']= STORIES.doc_str.str.strip()


In [13]:
STORIES.to_csv(f"{output_dir}/pg2591-STORIES.csv", index=True)

## Split Stories into Paragraphs

In [14]:
para_pat = r'\n\n+'

In [15]:
OHCO[:2]

['doc_title', 'para_num']

In [16]:
PARAS =STORIES['doc_str'].str.split(para_pat, expand=True).stack()\
    .to_frame('para_str').sort_index()
PARAS.index.names = OHCO[:2]
PARAS

para_str
doc_title para_num                                                   
ASHPUTTEL 0         The wife of a rich man fell sick; and when she...
          1         There she was forced to do hard work; to rise ...
          2         It happened once that the father was going to ...
          3         Now it happened that the king of that land hel...
          4         Then she threw the peas down among the ashes, ...
...                                                               ...
TOM THUMB 17        The wolf did not want to be asked twice; so th...
          18        This was just what Tom had reckoned upon; and ...
          19        The woodman and his wife, being awakened by th...
          20        ‘Well,’ said they, ‘you are come back, and we ...
          21        Then they hugged and kissed their dear little ...

[906 rows x 1 columns]

In [17]:
output_dir

'../working'

In [18]:
PARAS.to_csv(f"{output_dir}/pg2591-PARAS.csv", index=True)

## Spliting Paragraphs into Sentences

In [19]:
sent_pat = r'[.?!;:]+'
SENTS = PARAS['para_str'].str.split(sent_pat, expand=True).stack()\
    .to_frame('sent_str')
SENTS.index.names = OHCO[:3]

In [20]:
SENTS = SENTS[~SENTS['sent_str'].str.match(r'^\s*$')] # Remove empty paragraphs
SENTS.sent_str = SENTS.sent_str.str.strip() # CRUCIAL TO REMOVE BLANK TOKENS

In [21]:
SENTS.head()

sent_str
doc_title para_num sentence_num                                                   
ASHPUTTEL 0        0                              The wife of a rich man fell sick
                   1             and when she felt that her end drew\nnigh, she...
                   2             ’ Soon\nafterwards she shut her eyes and died,...
                   3             and the little girl went every day to her grav...
                   4             And the snow fell and spread a beautiful\nwhit...

## Spliting Senetences into Tokens

In [22]:
token_pat = r"[\s',-]+"
TOKENS = SENTS['sent_str'].str.split(token_pat, expand=True).stack()\
    .to_frame('token_str')

In [23]:
keep_whitespace = True

if keep_whitespace:
    # Return a tokenized copy of text
    # using NLTK's recommended word tokenizer.
    TOKENS = SENTS.sent_str\
            .apply(lambda x: pd.Series(nltk.pos_tag(nltk.word_tokenize(x))))\
            .stack()\
            .to_frame('pos_tuple')
else:
    # Tokenize a string on whitespace (space, tab, newline).
    # In general, users should use the string ``split()`` method instead.
    # Returns fewer tokens.
    TOKENS = SENTS.sent_str\
            .apply(lambda x: pd.Series(nltk.pos_tag(nltk.WhitespaceTokenizer().tokenize(x))))\
            .stack()\
            .to_frame('pos_tuple')


In [24]:
TOKENS.index.names = OHCO
TOKENS

pos_tuple
doc_title para_num sentence_num token_num             
ASHPUTTEL 0        0            0            (The, DT)
                                1           (wife, NN)
                                2             (of, IN)
                                3              (a, DT)
                                4           (rich, JJ)
...                                                ...
TOM THUMB 21       3            40            (s, VBZ)
                                41            (no, DT)
                                42         (place, NN)
                                43          (like, IN)
                                44          (HOME, NN)

[115019 rows x 1 columns]

In [25]:
# TOKENS.index.names = OHCO[:4]

In [26]:
# TOKENS['term_str'] = TOKENS.token_str.replace(r'[\W_]+', '', regex=True).str.lower()
# TOKENS.head()

In [27]:
TOKENS['pos'] = TOKENS.pos_tuple.apply(lambda x: x[1])
TOKENS['token_str'] = TOKENS.pos_tuple.apply(lambda x: x[0])
TOKENS['term_str'] = TOKENS.token_str.str.lower().str.replace(r"\W+", "", regex=True)
TOKENS['pos_group'] = TOKENS.pos.str[:2]
TOKENS.head()


pos_tuple pos token_str term_str  \
doc_title para_num sentence_num token_num                                      
ASHPUTTEL 0        0            0           (The, DT)  DT       The      the   
                                1          (wife, NN)  NN      wife     wife   
                                2            (of, IN)  IN        of       of   
                                3             (a, DT)  DT         a        a   
                                4          (rich, JJ)  JJ      rich     rich   

                                          pos_group  
doc_title para_num sentence_num token_num            
ASHPUTTEL 0        0            0                DT  
                                1                NN  
                                2                IN  
                                3                DT  
                                4                JJ

##  Make a VOCAB Table

In [28]:
VOCAB = TOKENS.term_str.value_counts().to_frame('n')
VOCAB.index.name = 'term_str'
VOCAB['p'] = VOCAB.n / VOCAB.n.sum()
VOCAB['i'] = -np.log2(VOCAB.p)
VOCAB['n_chars'] = VOCAB.index.str.len()
VOCAB['max_pos_group'] = TOKENS[['term_str','pos_group']].value_counts().unstack(fill_value=0).idxmax(1)
VOCAB['max_pos'] = TOKENS[['term_str','pos']].value_counts().unstack(fill_value=0).idxmax(1)
VOCAB

,n,p,i,n_chars,max_pos_group,max_pos
term_str,,,,,,
,13973,0.121484,3.041158,0,",",","
the,6927,0.060225,4.053498,3,DT,DT
and,5433,0.047236,4.403979,3,CC,CC
to,2662,0.023144,5.433218,2,TO,TO
he,2093,0.018197,5.780156,2,PR,PRP
...,...,...,...,...,...,...
beef,1,0.000009,16.811513,4,NN,NN
ham,1,0.000009,16.811513,3,NN,NN
crawl,1,0.000009,16.811513,5,VB,VB


###  Adding P Stem

In [29]:
from nltk.stem.porter import PorterStemmer
p_stemmer = PorterStemmer()
VOCAB['p_stem'] = [p_stemmer.stem(term_str) for term_str in VOCAB.index]
VOCAB


,n,p,i,n_chars,max_pos_group,max_pos,p_stem
term_str,,,,,,,
,13973,0.121484,3.041158,0,",",",",
the,6927,0.060225,4.053498,3,DT,DT,the
and,5433,0.047236,4.403979,3,CC,CC,and
to,2662,0.023144,5.433218,2,TO,TO,to
he,2093,0.018197,5.780156,2,PR,PRP,he
...,...,...,...,...,...,...,...
beef,1,0.000009,16.811513,4,NN,NN,beef
ham,1,0.000009,16.811513,3,NN,NN,ham
crawl,1,0.000009,16.811513,5,VB,VB,crawl


### Adding Stop Words

In [30]:
sw = pd.DataFrame({'stop': 1}, index=nltk.corpus.stopwords.words('english'))
sw.index.name='term_str'
sw.head()
if 'stop' not in VOCAB.columns:
    VOCAB = VOCAB.join(sw)
    VOCAB['stop'] = VOCAB['stop'].fillna(0).astype('int')
VOCAB.head()

,n,p,i,n_chars,max_pos_group,max_pos,p_stem,stop
term_str,,,,,,,,
,13973,0.121484,3.041158,0,",",",",,0
the,6927,0.060225,4.053498,3,DT,DT,the,1
and,5433,0.047236,4.403979,3,CC,CC,and,1
to,2662,0.023144,5.433218,2,TO,TO,to,1
he,2093,0.018197,5.780156,2,PR,PRP,he,1


## Saving Files

In [31]:
TOKENS.to_csv(csv_file)

In [32]:
VOCAB.to_csv(f"{output_dir}/pg2591-VOCAB.csv", index=True)